In [117]:
# for reading env file
from dotenv import load_dotenv
import os

# for loading db
import pymysql

# for working part
import numpy as np
import pandas as pd

# for config saving
import json

# for parquet files
import fastparquet

In [94]:
load_dotenv()

conn = pymysql.connect(host=os.getenv('host'),
                       port=int(os.getenv('port')),
                       user=os.getenv('user'),
                       password=os.getenv('password'),
                       database=os.getenv('database'),
                       ssl={'ssl_mode':'REQUIRED'})

cursor = conn.cursor()
database_name = os.getenv('database')
print(f'database connected, database name is {database_name}')

database connected, database name is pharma_db


In [95]:
with open('../config/selected_tables.json') as f :
    tables_config = json.load(f)

with open('../config/selected_columns.json') as f :
    columns_config = json.load(f)

In [96]:
def load_tables(selected_tables,columns_config):
    data = {}

    for table in selected_tables:
        cols = ', '.join(columns_config[table])

        query = f'select {cols} from {table}'

        data[table] = pd.read_sql(query,conn)

    return data

In [97]:
selected_tables = tables_config['selected_tables']

data = load_tables(selected_tables,columns_config)

C:\Users\hp5cd\AppData\Local\Temp\ipykernel_6088\1897608191.py:9: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  data[table] = pd.read_sql(query,conn)


In [100]:
data.keys()

dict_keys(['contacts', 'leads', 'order_items', 'orders', 'pii', 'products', 'order_history', 'bulk', 'packaging_cost', 'packaging_master', 'product_cost', 'product_packaging'])

In [101]:
contacts = data['contacts']
leads = data['leads']
order_items = data['order_items']
orders = data['orders']
pii = data['pii']
products = data['products']
order_history = data['order_history']
bulk = data['bulk']
packaging_cost = data['packaging_cost']
packaging_master = data['packaging_master']
product_cost = data['product_cost']
product_packaging = data['product_packaging']


In [102]:
for i,j in data.items():
    print(i)
    print(j.columns)
    print()


contacts
Index(['contact_id', 'pii_id', 'profile', 'profile_details'], dtype='object')

leads
Index(['lead_id', 'pii_id', 'owner', 'lead_source', 'assigned_date',
       'follow_up_date'],
      dtype='object')

order_items
Index(['order_id', 'product_sku', 'quantity', 'item_price', 'line_item_total',
       'exchange_rate'],
      dtype='object')

orders
Index(['order_id', 'pii_id', 'payment_mode', 'agent_id', 'channel',
       'order_date', 'order_status'],
      dtype='object')

pii
Index(['pii_id', 'country', 'state', 'pincode'], dtype='object')

products
Index(['sku', 'bulk_sku', 'variation_value', 'variation_uom', 'itemQty',
       'gross_weight', 'net_weight'],
      dtype='object')

order_history
Index(['order_id', 'status', 'status_changed_time', 'status_changed_by'], dtype='object')

bulk
Index(['sku', 'uom', 'tax_percent', 'category', 'generic_name'], dtype='object')

packaging_cost
Index(['packaging_sku', 'cost_per_unit', 'is_active'], dtype='object')

packaging_master
Inde

In [103]:
master_datset = order_items[['order_id','product_sku', 'quantity', 'item_price',
                                           'line_item_total','exchange_rate']].merge(
                                               
                                               orders[['order_id', 'pii_id', 'channel','order_date',
                                                        'order_status']],how='left',on='order_id').merge(

                                                   pii[['pii_id','country', 'state',
                                                         'pincode']],how='left',on='pii_id').merge(
                                                             
                                                       contacts[['pii_id','profile','profile_details']],how='left',on='pii_id')

In [104]:
productcost = products[['sku', 'bulk_sku','net_weight']].merge(product_cost[['bulk_sku', 'cost_per_kg_or_litre']],how='left',on='bulk_sku')

productcost['cost_per_pc'] = productcost['cost_per_kg_or_litre'] * productcost['net_weight']

packingcost = product_packaging[['product_sku', 'packaging_sku']].merge(
    packaging_cost[packaging_cost['is_active']==True][['packaging_sku', 'cost_per_unit']])

packingcost = packingcost.groupby('product_sku',as_index=False)['cost_per_unit'].sum()


final_cost = productcost[['sku','cost_per_pc']].merge(packingcost[['product_sku','cost_per_unit']],
                                                      how='left',left_on = 'sku',right_on = 'product_sku')

final_cost['base_cost'] = final_cost['cost_per_pc'] + final_cost['cost_per_unit']
final_cost = final_cost[['sku','base_cost']]

In [105]:
master_datset =  master_datset.merge(final_cost,how='left',left_on='product_sku', right_on='sku')

In [107]:
master_datset = master_datset.merge(leads[['pii_id','lead_source']],how='left',on='pii_id')

In [112]:
productdetails = products[['sku','bulk_sku']].merge(bulk[['sku','category','generic_name']],how='left',left_on='bulk_sku',right_on='sku')
productdetails = productdetails[['sku_x','category','generic_name']]

master_datset = master_datset.merge(productdetails[['sku_x','category','generic_name']],how='left',left_on='sku',right_on='sku_x')


In [113]:
master_datset.columns

Index(['order_id', 'product_sku', 'quantity', 'item_price', 'line_item_total',
       'exchange_rate', 'pii_id', 'channel', 'order_date', 'order_status',
       'country', 'state', 'pincode', 'profile', 'profile_details', 'sku',
       'base_cost', 'lead_source', 'sku_x', 'category', 'generic_name'],
      dtype='object')

In [114]:
master_datset = master_datset[['order_id', 'product_sku', 'quantity', 'item_price',
                                'line_item_total','exchange_rate', 'pii_id', 'channel',
                                  'order_date', 'order_status','country', 'state', 'pincode',
                                    'profile', 'profile_details','base_cost', 'lead_source',
                                    'category', 'generic_name']]

In [115]:
master_datset['order_date'] = pd.to_datetime(master_datset['order_date'],errors='coerce')

In [118]:
master_datset.to_parquet("E:\\Python New\\revenue-intelligence\\data\\raw_data_files\\master_datset.parquet",engine="fastparquet",index=False)